In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Datasets/IEOR4578_Forecasting/aggregate_hourly.csv" "/content/aggregate_hourly.csv"

In [3]:
!pip install statsforecast

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 5.2 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [4]:
from pathlib import Path

# Input file
AGG_PATH = "/content/aggregate_hourly.csv"

# Model
SEASON_LENGTH  = 24   # hourly data → daily seasonality
LOOKBACK_HOURS = 672  # 4 weeks lookback
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Data split boundaries
TRAIN_END  = "2013-12-31 23:00:00"
VAL_START  = "2014-01-01 00:00:00"
VAL_END    = "2014-04-30 23:00:00"
TEST_START = "2014-05-01 00:00:00"

In [5]:
import warnings
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

In [6]:
# ── PREPARE DATA ──────────────────────────────────────────────────────────────
# aggregate_hourly.csv: single time series (total load across all 156 clients)
df = pd.read_csv(AGG_PATH, parse_dates=["timestamp"])

# StatsForecast expects: unique_id, ds, y
df = df.rename(columns={"timestamp": "ds", "aggregate_hourly_kW": "y"})
df["unique_id"] = "aggregate"
df = df[["unique_id", "ds", "y"]].sort_values("ds").reset_index(drop=True)

# Time-based split
train = df[df["ds"] <= TRAIN_END].copy()
val   = df[(df["ds"] >= VAL_START) & (df["ds"] <= VAL_END)].copy()
test  = df[df["ds"] >= TEST_START].copy()

# Apply 4-week lookback on train
train = train.tail(LOOKBACK_HOURS).reset_index(drop=True)

print(f"train : {train.shape}  {train['ds'].min()} → {train['ds'].max()}")
print(f"val   : {val.shape}    {val['ds'].min()} → {val['ds'].max()}")
print(f"test  : {test.shape}   {test['ds'].min()} → {test['ds'].max()}")

train : (672, 3)  2013-12-04 00:00:00 → 2013-12-31 23:00:00
val   : (2856, 3)    2014-01-01 00:00:00 → 2014-04-30 23:00:00
test  : (5856, 3)   2014-05-01 00:00:00 → 2014-12-31 23:00:00


In [7]:
# ── EVALUATION METRICS ────────────────────────────────────────────────────────
def compute_metrics(df, target_col="y", pred_col="AutoARIMA"):
    y    = df[target_col].values
    yhat = df[pred_col].values
    mse  = np.mean((y - yhat) ** 2)
    mae  = np.mean(np.abs(y - yhat))
    wape = np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))
    return {"MSE": round(mse, 4), "MAE": round(mae, 4), "WAPE": round(wape, 4)}

In [8]:
# ── TRAIN AutoARIMA (Aggregate Benchmark) ─────────────────────────────────────
model = AutoARIMA(
    season_length=SEASON_LENGTH,
    approximation=True,
    stepwise=True
)

sf = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf.fit(train[["unique_id", "ds", "y"]])

joblib.dump(sf, MODEL_DIR / "autoarima_agg_val.joblib")
print("Training complete. Model saved → models/autoarima_agg_val.joblib")

Training complete. Model saved → models/autoarima_agg_val.joblib


In [9]:
# ── VALIDATION FORECAST ───────────────────────────────────────────────────────
val_h = val["ds"].nunique()

val_preds = sf.predict(h=val_h)
val_eval  = val[["unique_id", "ds", "y"]].merge(val_preds, on=["unique_id", "ds"])

val_metrics = compute_metrics(val_eval)
print("── Validation Metrics ──")
print(val_metrics)

── Validation Metrics ──
{'MSE': np.float64(147666054.8937), 'MAE': np.float64(9359.3673), 'WAPE': np.float64(0.0978)}


In [10]:
# ── TEST EVALUATION ───────────────────────────────────────────────────────────
# Retrain on train+val (with lookback), then predict on held-out test set
train_val = pd.concat([train, val], ignore_index=True).sort_values("ds")
train_val = train_val.tail(LOOKBACK_HOURS).reset_index(drop=True)

model = AutoARIMA(
    season_length=SEASON_LENGTH,
    approximation=True,
    stepwise=True
)

sf_final = StatsForecast(models=[model], freq="h", n_jobs=-1)
sf_final.fit(train_val[["unique_id", "ds", "y"]])

joblib.dump(sf_final, MODEL_DIR / "autoarima_agg_final.joblib")
print("Final model saved → models/autoarima_agg_final.joblib")

test_h     = test["ds"].nunique()
test_preds = sf_final.predict(h=test_h)
test_eval  = test[["unique_id", "ds", "y"]].merge(test_preds, on=["unique_id", "ds"])

test_metrics = compute_metrics(test_eval)
print("── Test Metrics ──")
print(test_metrics)

Final model saved → models/autoarima_agg_final.joblib
── Test Metrics ──
{'MSE': np.float64(203562485.1573), 'MAE': np.float64(10726.1767), 'WAPE': np.float64(0.1004)}
